# ☕🗺️ Mapify Extended: Google Maps Cafe Photo & Metadata Harvester

This notebook clones the production codebase directly from [GitHub (osumy/mapify)](https://github.com/osumy/mapify) and runs the zero-overhead, anti-bot hardened Google Maps scraper across pre-matched venues.

### 🌟 Extracted Fields:
- **Canonical Identifiers:** `place_id`, `cid`, `fid`, `kgmid`, `google_maps_url`
- **Comprehensive Metadata (Level 1):** `title`, `category_name`, `total_score` (rating), `reviews_count`, `address`, `phone`, `phone_unformatted`, `website`, `plus_code`, `is_claimed`, `is_permanently_closed`, `is_temporarily_closed`, `price_level`, `opening_hours`, `popular_times`
- **High-Resolution Photos:** Owner/verified uploads vs customer UGC gallery photos with original resolution (`=s0-rw`)

### 🚀 Quickstart Checklist:
1. **Enable Internet:** In the right sidebar (**Notebook Options**), ensure **Internet -> Internet On** is enabled.
2. **Attach Input Shard:** Click **+ Add Input** and attach your cafe shard database (e.g. `matched_shard_01...db`).
3. **Run in Background:** Click **Save Version** -> **Save & Run All (Commit)**.
4. **Download Results:** Once complete (~45-60 min for 533 cafes), download `cafe_photos.db` from the **Output** tab.

In [ ]:
# Step 1: Clone Mapify repository from GitHub & install dependencies
import os
import sys

# Ensure working directory is cleanly anchored at /kaggle/working
if os.path.exists('/kaggle/working'):
    os.chdir('/kaggle/working')

!if [ ! -d 'mapify' ]; then git clone https://github.com/osumy/mapify.git; else (cd mapify && git pull); fi
%cd /kaggle/working/mapify

!pip install -q -r requirements.txt
!playwright install chromium
!playwright install-deps chromium
print('✅ Mapify environment, repository, and Playwright Chromium ready!')

In [ ]:
# Step 2: Import modular Mapify components and configure input/output paths
import sys
import os
import glob
from pathlib import Path

# Ensure mapify root is on sys.path for direct imports
repo_dir = '/kaggle/working/mapify' if os.path.exists('/kaggle/working/mapify') else os.getcwd()
if repo_dir not in sys.path:
    sys.path.insert(0, repo_dir)

from src.config import ScraperConfig
from src.db import DatabaseManager
from src.scraper import MapifyScraper
from run import run_pipeline

# Auto-detect SQLite shard database from Kaggle input directory
input_dbs = sorted(glob.glob('/kaggle/input/**/*.db', recursive=True))
if input_dbs:
    source_db = input_dbs[0]
    print(f'🔍 Auto-detected input database: {source_db}')
    if len(input_dbs) > 1:
        print(f'   (Found {len(input_dbs)} databases, using: {source_db})')
else:
    source_db = 'matched_shard_01_of_08_534cafes.db'
    print(f'⚠️ Fallback source path: {source_db}')

output_db = '/kaggle/working/cafe_photos.db'

config = ScraperConfig(
    SOURCE_DB_PATH=source_db,
    OUTPUT_DB_PATH=output_db,
    HEADLESS=True,
    CAFES_PER_BATCH=35,
    BATCH_COOLDOWN_SECONDS=120.0,
    MIN_DELAY_PER_CAFE=4.5,
    MAX_DELAY_PER_CAFE=8.5,
)

print('✅ ScraperConfig and modular components initialized successfully!')

In [ ]:
# Step 3: Run Modular Harvester Pipeline with Direct Place URL, Pacing & Checkpointing
import asyncio

async def run_harvester():
    db = DatabaseManager(config.SOURCE_DB_PATH, config.OUTPUT_DB_PATH)
    pending = db.load_pending_cafes()
    total = len(pending)

    print('=' * 60)
    print(f'🚀 MAPIFY EXTENDED HARVESTER RUNNING ON KAGGLE')
    print(f'Source DB: {config.SOURCE_DB_PATH}')
    print(f'Output DB: {config.OUTPUT_DB_PATH}')
    print(f'Pending Cafes to Scrape: {total}')
    print('=' * 60)

    if total == 0:
        print('✅ All cafes in this database are already scraped!')
        return

    scraper = MapifyScraper(config=config, db_manager=db)
    await scraper.start()

    try:
        stats = await run_pipeline(scraper=scraper, pending_cafes=pending, config=config)
    finally:
        await scraper.close()

    print('=' * 60)
    print('📊 SESSION COMPLETE SUMMARY')
    print(f'Total Processed: {stats.get("matched", 0) + stats.get("not_found", 0) + stats.get("errors", 0)} / {total}')
    print(f'Matched: {stats.get("matched", 0)}')
    print(f'Not Found: {stats.get("not_found", 0)}')
    print(f'Errors: {stats.get("errors", 0)}')
    print(f'Total High-Res Photos Harvested: {stats.get("photos", 0)}')
    print(f'Owner / Verified Photos: {stats.get("owner_photos", 0)}')
    print(f'Output Database: {config.OUTPUT_DB_PATH}')
    print('=' * 60)

# Run async pipeline in Kaggle
await run_harvester()

In [ ]:
# Step 4: Inspect Checkpoint Status, Extended Metadata & Photo Statistics
import sqlite3
import pandas as pd

conn = sqlite3.connect(config.OUTPUT_DB_PATH)

print('📊 Shard Checkpoint & Run Status:')
try:
    df_cp = pd.read_sql_query('''
    SELECT shard_name, last_cafe_id, last_cafe_name, total_processed, matched_count, not_found_count, error_count, total_photos, status, updated_at
    FROM checkpoints;
    ''', conn)
    display(df_cp)
except Exception:
    pass

print('\n📌 Sample Venues with Level 1 Metadata:')
df_places = pd.read_sql_query('''
SELECT cafe_id, title, category_name, total_score, reviews_count, phone, website, is_claimed, photos_count
FROM places
WHERE status = 'matched'
LIMIT 5;
''', conn)
display(df_places)

print('\n📸 Sample High-Res Photos (Owner vs Customer UGC):')
df_photos = pd.read_sql_query('''
SELECT cafe_id, is_owner_photo, source_type, author_name, high_res_url
FROM place_photos
ORDER BY is_owner_photo DESC
LIMIT 10;
''', conn)
display(df_photos)

total_places = conn.execute('SELECT count(*) FROM places WHERE status = "matched"').fetchone()[0]
with_score = conn.execute('SELECT count(*) FROM places WHERE total_score IS NOT NULL').fetchone()[0]
with_phone = conn.execute('SELECT count(*) FROM places WHERE phone IS NOT NULL').fetchone()[0]
with_web = conn.execute('SELECT count(*) FROM places WHERE website IS NOT NULL').fetchone()[0]
with_addr = conn.execute('SELECT count(*) FROM places WHERE address IS NOT NULL').fetchone()[0]
total_photos = conn.execute('SELECT count(*) FROM place_photos').fetchone()[0]
owner_photos = conn.execute('SELECT count(*) FROM place_photos WHERE is_owner_photo = 1').fetchone()[0]

print('\n' + '=' * 60)
print('📊 METADATA HARVEST COVERAGE:')
print(f'✅ Total Matched Venues: {total_places}')
print(f'✅ With Ratings & Review Counts: {with_score} ({round(with_score/max(1, total_places)*100, 1)}%)')
print(f'✅ With Phone Numbers: {with_phone} ({round(with_phone/max(1, total_places)*100, 1)}%)')
print(f'✅ With Websites: {with_web} ({round(with_web/max(1, total_places)*100, 1)}%)')
print(f'✅ With Addresses: {with_addr} ({round(with_addr/max(1, total_places)*100, 1)}%)')
print(f'✅ Total High-Res Photos: {total_photos} ({owner_photos} Owner/Verified, {total_photos - owner_photos} UGC)')
print('=' * 60)
conn.close()